In [91]:
pip install gurobipy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Conjuntos

In [92]:
from gurobipy import Model, GRB, quicksum

# Datos del problema
regiones = ["R1", "R2", "R3", "R4", "R5", "R6"]
ciudades = ["Antofagasta", "Valparaíso", "Santiago", "Rancagua", "Concepción", "Puerto Montt"]
tipos_planta = ["Pequeña", "Grande"]
transportes = ["AT1", "AT2", "AT3"]
años = [1, 2, 3]

Parametros

In [93]:
demanda_actual = {
    "R1": 951776, "R2": 967364, "R3": 512051, "R4": 386248, "R5": 946174, "R6": 303445
}
tasas_crecimiento = {
    "R1": 0.16, "R2": 0.22, "R3": 0.26, "R4": 0.15, "R5": 0.39, "R6": 0.30
}
capacidad_planta = {"Pequeña": 4636446, "Grande": 14966773}
costos_fijos = {
    ("Antofagasta", "Pequeña"): 18236639, ("Antofagasta", "Grande"): 60788796,
    ("Valparaíso", "Pequeña"): 8838286, ("Valparaíso", "Grande"): 32734393,
    ("Santiago", "Pequeña"): 6840758, ("Santiago", "Grande"): 32575039,
    ("Rancagua", "Pequeña"): 13378246, ("Rancagua", "Grande"): 53512984,
    ("Concepción", "Pequeña"): 26394217, ("Concepción", "Grande"): 65985543,
    ("Puerto Montt", "Pequeña"): 3678737, ("Puerto Montt", "Grande"): 26276695
}
costos_variables = {
    ("Antofagasta", "Pequeña"): 28.20, ("Antofagasta", "Grande"): 28.20,
    ("Valparaíso", "Pequeña"): 41.68, ("Valparaíso", "Grande"): 41.68,
    ("Santiago", "Pequeña"): 18.57, ("Santiago", "Grande"): 18.57,
    ("Rancagua", "Pequeña"): 17.68, ("Rancagua", "Grande"): 17.68,
    ("Concepción", "Pequeña"): 50.11, ("Concepción", "Grande"): 50.11,
    ("Puerto Montt", "Pequeña"): 43.55, ("Puerto Montt", "Grande"): 43.55
}
costos_apertura = {
    ("Antofagasta", "Pequeña"): 86626147, ("Antofagasta", "Grande"): 201456157,
    ("Valparaíso", "Pequeña"): 115721215, ("Valparaíso", "Grande"): 199519337,
    ("Santiago", "Pequeña"): 172235977, ("Santiago", "Grande"): 291925385,
    ("Rancagua", "Pequeña"): 0, ("Rancagua", "Grande"): 299031830,
    ("Concepción", "Pequeña"): 57494934, ("Concepción", "Grande"): 179671671,
    ("Puerto Montt", "Pequeña"): 175561277, ("Puerto Montt", "Grande"): 337617842
}
costos_transporte = {
    ("AT1", "Antofagasta", "R1"): 1.06, ("AT1", "Antofagasta", "R2"): 2.80, ("AT1", "Antofagasta", "R3"): 10.29,
    ("AT1", "Antofagasta", "R4"): 4.87, ("AT1", "Antofagasta", "R5"): 6.41, ("AT1", "Antofagasta", "R6"): 10.35,
    ("AT1", "Valparaíso", "R1"): 3.49, ("AT1", "Valparaíso", "R2"): 6.19, ("AT1", "Valparaíso", "R3"): 3.39,
    ("AT1", "Valparaíso", "R4"): 6.77, ("AT1", "Valparaíso", "R5"): 3.07, ("AT1", "Valparaíso", "R6"): 6.61,
    ("AT1", "Santiago", "R1"): 6.38, ("AT1", "Santiago", "R2"): 5.88, ("AT1", "Santiago", "R3"): 5.63,
    ("AT1", "Santiago", "R4"): 1.01, ("AT1", "Santiago", "R5"): 3.15, ("AT1", "Santiago", "R6"): 5.67,
    ("AT1", "Rancagua", "R1"): 3.44, ("AT1", "Rancagua", "R2"): 1.48, ("AT1", "Rancagua", "R3"): 2.79,
    ("AT1", "Rancagua", "R4"): 2.80, ("AT1", "Rancagua", "R5"): 5.30, ("AT1", "Rancagua", "R6"): 1.29,
    ("AT1", "Concepción", "R1"): 5.94, ("AT1", "Concepción", "R2"): 7.33, ("AT1", "Concepción", "R3"): 1.80,
    ("AT1", "Concepción", "R4"): 9.48, ("AT1", "Concepción", "R5"): 2.82, ("AT1", "Concepción", "R6"): 8.25,
    ("AT1", "Puerto Montt", "R1"): 2.57, ("AT1", "Puerto Montt", "R2"): 9.63, ("AT1", "Puerto Montt", "R3"): 4.84,
    ("AT1", "Puerto Montt", "R4"): 6.64, ("AT1", "Puerto Montt", "R5"): 6.48, ("AT1", "Puerto Montt", "R6"): 8.54,
    ("AT2", "Antofagasta", "R1"): 10.03, ("AT2", "Antofagasta", "R2"): 4.09, ("AT2", "Antofagasta", "R3"): 4.55,
    ("AT2", "Antofagasta", "R4"): 7.84, ("AT2", "Antofagasta", "R5"): 5.33, ("AT2", "Antofagasta", "R6"): 10.63,
    ("AT2", "Valparaíso", "R1"): 10.52, ("AT2", "Valparaíso", "R2"): 1.82, ("AT2", "Valparaíso", "R3"): 3.91,
    ("AT2", "Valparaíso", "R4"): 8.20, ("AT2", "Valparaíso", "R5"): 5.88, ("AT2", "Valparaíso", "R6"): 2.33,
    ("AT2", "Santiago", "R1"): 1.90, ("AT2", "Santiago", "R2"): 8.89, ("AT2", "Santiago", "R3"): 6.55,
    ("AT2", "Santiago", "R4"): 9.71, ("AT2", "Santiago", "R5"): 7.03, ("AT2", "Santiago", "R6"): 10.23,
    ("AT2", "Rancagua", "R1"): 2.06, ("AT2", "Rancagua", "R2"): 10.17, ("AT2", "Rancagua", "R3"): 2.12,
    ("AT2", "Rancagua", "R4"): 6.11, ("AT2", "Rancagua", "R5"): 3.79, ("AT2", "Rancagua", "R6"): 6.19,
    ("AT2", "Concepción", "R1"): 2.54, ("AT2", "Concepción", "R2"): 6.95, ("AT2", "Concepción", "R3"): 8.57,
    ("AT2", "Concepción", "R4"): 10.50, ("AT2", "Concepción", "R5"): 4.85, ("AT2", "Concepción", "R6"): 5.31,
    ("AT2", "Puerto Montt", "R1"): 7.92, ("AT2", "Puerto Montt", "R2"): 10.32, ("AT2", "Puerto Montt", "R3"): 1.41,
    ("AT2", "Puerto Montt", "R4"): 4.94, ("AT2", "Puerto Montt", "R5"): 2.74, ("AT2", "Puerto Montt", "R6"): 8.08,
    ("AT3", "Antofagasta", "R1"): 9.86, ("AT3", "Antofagasta", "R2"): 4.30, ("AT3", "Antofagasta", "R3"): 8.10,
    ("AT3", "Antofagasta", "R4"): 9.63, ("AT3", "Antofagasta", "R5"): 7.40, ("AT3", "Antofagasta", "R6"): 6.47,
    ("AT3", "Valparaíso", "R1"): 1.58, ("AT3", "Valparaíso", "R2"): 2.71, ("AT3", "Valparaíso", "R3"): 3.08,
    ("AT3", "Valparaíso", "R4"): 5.91, ("AT3", "Valparaíso", "R5"): 7.99, ("AT3", "Valparaíso", "R6"): 5.11,
    ("AT3", "Santiago", "R1"): 9.13, ("AT3", "Santiago", "R2"): 10.03, ("AT3", "Santiago", "R3"): 6.77,
    ("AT3", "Santiago", "R4"): 5.70, ("AT3", "Santiago", "R5"): 3.62, ("AT3", "Santiago", "R6"): 8.58,
    ("AT3", "Rancagua", "R1"): 8.95, ("AT3", "Rancagua", "R2"): 7.37, ("AT3", "Rancagua", "R3"): 10.29,
    ("AT3", "Rancagua", "R4"): 3.34, ("AT3", "Rancagua", "R5"): 2.21, ("AT3", "Rancagua", "R6"): 4.58,
    ("AT3", "Concepción", "R1"): 9.62, ("AT3", "Concepción", "R2"): 3.78, ("AT3", "Concepción", "R3"): 5.19,
    ("AT3", "Concepción", "R4"): 2.61, ("AT3", "Concepción", "R5"): 3.19, ("AT3", "Concepción", "R6"): 1.78,
    ("AT3", "Puerto Montt", "R1"): 10.32, ("AT3", "Puerto Montt", "R2"): 8.88, ("AT3", "Puerto Montt", "R3"): 10.87,
    ("AT3", "Puerto Montt", "R4"): 10.38, ("AT3", "Puerto Montt", "R5"): 5.83, ("AT3", "Puerto Montt", "R6"): 1.54
}


Variables de Decision

In [94]:
# Crear el modelo
m = Model("FunnysCompany")

# 1. Variables de decisión
x = m.addVars(ciudades, tipos_planta, años, vtype=GRB.BINARY, name="x")
y = m.addVars(ciudades, regiones, transportes, años, vtype=GRB.CONTINUOUS, name="y")
w = m.addVars(ciudades, transportes, vtype=GRB.BINARY, name="w")

# Funcion Objetivo

In [95]:
# 2. Función Objetivo
m.setObjective(
    quicksum(
        (costos_apertura[(i, j)] + costos_fijos[(i, j)]) * x[i, j, t]
        for i in ciudades for j in tipos_planta for t in años
    ) +
    quicksum(
        costos_variables[(i, "Pequeña")] * y[i, k, f, t]
        for i in ciudades for k in regiones for f in transportes for t in años
    ) +
    quicksum(
        costos_transporte[(f, i, k)] * y[i, k, f, t]
        for i in ciudades for k in regiones for f in transportes for t in años
    ),
    GRB.MINIMIZE
)

## Restricciones

In [96]:
# R1: No abrir más de una vez la misma planta.
m.addConstrs((quicksum(x[i, j, t] for t in años) <= 1 
              for i in ciudades for j in tipos_planta), name="R1")

# R2: Satisfacer demanda estricta con crecimiento.
m.addConstrs((quicksum(y[i, k, f, t] for i in ciudades for f in transportes) ==
              demanda_actual[k] * (1 + tasas_crecimiento[k])**(t-1)
              for k in regiones for t in años), name="R2")

# R3: Solo puede haber un tipo de planta por ciudad.
m.addConstrs((quicksum(x[i, j, t] for j in tipos_planta for t in años) <= 1 
              for i in ciudades), name="R3")

# R4: La cantidad de plantas no debe ser superior a 5.
m.addConstr(quicksum(x[i, j, t] for i in ciudades for j in tipos_planta for t in años) <= 5, name="R4")

# R5: La producción no supera la capacidad instalada acumulada.
m.addConstrs((quicksum(y[i, k, f, t] for k in regiones for f in transportes) <=
              quicksum(capacidad_planta[j] * quicksum(x[i, j, tau] for tau in años if tau <= t)
                       for j in tipos_planta)
              for i in ciudades for t in años), name="R5")

# R_Rancagua: Fija la existencia de la planta pequeña operativa desde el año 1.
m.addConstr(x["Rancagua", "Pequeña", 1] >= 1, name="R_Rancagua_Base")

# R6: Cada ciudad puede usar como MÁXIMO 1 tipo de transporte en total.
m.addConstrs((quicksum(w[i, f] for f in transportes) <= 1 
              for i in ciudades), name="R6_UnicoTransporte")

# R7: Restricción Big-M. Solo hay envíos si el transporte 'f' fue elegido por la ciudad 'i'.
# BIG_M escalado a la máxima producción teórica (5 plantas grandes).
BIG_M = 14966773 * 5
m.addConstrs((y[i, k, f, t] <= BIG_M * w[i, f]
              for i in ciudades for k in regiones for f in transportes for t in años), name="R7_Link_yw")

# R8: Un transporte solo puede activarse si se construyó alguna planta en esa ciudad.
m.addConstrs((w[i, f] <= quicksum(x[i, j, t] for j in tipos_planta for t in años)
              for i in ciudades for f in transportes), name="R8_Vincular_W_X")

{('Antofagasta', 'AT1'): <gurobi.Constr *Awaiting Model Update*>,
 ('Antofagasta', 'AT2'): <gurobi.Constr *Awaiting Model Update*>,
 ('Antofagasta', 'AT3'): <gurobi.Constr *Awaiting Model Update*>,
 ('Valparaíso', 'AT1'): <gurobi.Constr *Awaiting Model Update*>,
 ('Valparaíso', 'AT2'): <gurobi.Constr *Awaiting Model Update*>,
 ('Valparaíso', 'AT3'): <gurobi.Constr *Awaiting Model Update*>,
 ('Santiago', 'AT1'): <gurobi.Constr *Awaiting Model Update*>,
 ('Santiago', 'AT2'): <gurobi.Constr *Awaiting Model Update*>,
 ('Santiago', 'AT3'): <gurobi.Constr *Awaiting Model Update*>,
 ('Rancagua', 'AT1'): <gurobi.Constr *Awaiting Model Update*>,
 ('Rancagua', 'AT2'): <gurobi.Constr *Awaiting Model Update*>,
 ('Rancagua', 'AT3'): <gurobi.Constr *Awaiting Model Update*>,
 ('Concepción', 'AT1'): <gurobi.Constr *Awaiting Model Update*>,
 ('Concepción', 'AT2'): <gurobi.Constr *Awaiting Model Update*>,
 ('Concepción', 'AT3'): <gurobi.Constr *Awaiting Model Update*>,
 ('Puerto Montt', 'AT1'): <gurobi.

## Resultados

In [97]:

m.Params.PoolSearchMode = 2  # Búsqueda rigurosa de las n mejores soluciones
m.Params.PoolSolutions = 2   # Retener las 2 mejores soluciones encontradas

# Ejecutar optimización
m.optimize()

# Verificamos si se encontró al menos una solución factible
if m.SolCount > 0:
    print(f"\n[INFO] El optimizador retuvo {m.SolCount} soluciones factibles en el Pool.")
    
    # Iteramos sobre las soluciones almacenadas (hasta un máximo de 2)
    for i in range(min(2, m.SolCount)):
        m.Params.SolutionNumber = i  # Fija el puntero en la solución i (0 = Óptima, 1 = Segunda mejor)
        
        jerarquia = "ÓPTIMA" if i == 0 else "SEGUNDA MEJOR"
        
        print(f"\n" + "="*60)
        print(f" SOLUCIÓN {jerarquia} (Opción {i+1}) | Costo Total: ${m.PoolObjVal:,.0f}")
        print("="*60)
        
        print("\n--- Decisiones de Infraestructura (X) y Transporte (W) ---")
        for i_ciudad in ciudades:
            for j_planta in tipos_planta:
                for t_year in años:
                    # Se utiliza .Xn para consultar el valor de la variable en la solución actual del Pool
                    if x[i_ciudad, j_planta, t_year].Xn > 0.5:
                        cantidad = int(round(x[i_ciudad, j_planta, t_year].Xn))
                        print(f"-> Se construye(n) {cantidad} planta(s) '{j_planta}' en {i_ciudad} (Año {t_year}).")
                        
                        # Buscar el transporte exclusivo asignado a esta ciudad
                        for f_transporte in transportes:
                            if w[i_ciudad, f_transporte].Xn > 0.5:
                                print(f"   Utilizará el medio de transporte: {f_transporte}")
                                
else:
    print("\n[ERROR] El modelo es infactible. No se encontraron soluciones.")

Set parameter PoolSearchMode to value 2
Set parameter PoolSolutions to value 2
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 5 5600 6-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Non-default parameters:
PoolSolutions  2
PoolSearchMode  2

Optimize a model with 404 rows, 378 columns and 1621 nonzeros (Min)
Model fingerprint: 0x583918bb
Model has 360 linear objective coefficients
Variable types: 324 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+07]
  Objective range  [2e+01, 4e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]

Presolve removed 18 rows and 6 columns
Presolve time: 0.00s
Presolved: 386 rows, 372 columns, 1521 nonzeros
Variable types: 324 continuous, 48 integer (48 binary)
Found heuristic solution: objective 8.233501e+08

Root relaxation: objective 3.963909e+08, 94 itera